# Agricultural Extension RAG — LightGBM → Cross-Encoder (bge-reranker)

Full pipeline, runs top to bottom (**Kernel → GPU T4 x 2 → Restart & Run All**):

1. Load data
2. nDCG@5 metric
3. Slot extractor (crop / issue / intent / zone) — shared parser for queries + doc titles
4. BM25 + TF-IDF
5. Rubric scorer (deterministic ceiling)
6. Candidate generation (BM25 top-K + slot injection) + feature builder
7. Recall check
8. LightGBM LambdaMART reranker + leak-free CV
9. Train final LightGBM
10. Cross-encoder (bge-reranker-base) fine-tune + `retrieve_ce`
11. **Leak-free CV of the full LightGBM → Cross-encoder pipeline**  ← the gate
12. Build submission

**Requires internet ON during Save & Run All** (downloads bge-reranker-base).
Read the cell-11 number before trusting the submission.

> **Why this design?** The system is a classic *retrieve-then-rerank* pipeline:
> a cheap lexical retriever (BM25) narrows the full document collection down to a
> manageable shortlist, then two progressively more expensive learned rerankers
> (LightGBM → cross-encoder) refine that shortlist into the final top-5 answer.
> Every stage is checked against a **leak-free cross-validation** score before
> being trusted, and a deterministic rubric baseline is used as a sanity ceiling
> throughout.


In [ ]:
# ==========================================================================
# 1. Imports + data load
# ==========================================================================
# Standard library / numeric stack used throughout the notebook.
import re, math, os                              # re/math: slot-extractor + BM25 math; os: env vars for CE stage
from pathlib import Path                         # convenient path handling for the Kaggle input directory
from collections import Counter, defaultdict     # Counter: term frequencies; defaultdict: BM25 inverted index
import numpy as np
import pandas as pd

# Default location of the competition data on Kaggle.
base = Path('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers')

# Fallback: if the exact competition path/slug doesn't exist (e.g. a different
# Kaggle mount point), search the whole /kaggle/input tree for documents.csv
# so the notebook still runs without manual path editing.
if not (base / 'documents.csv').exists():
    cands = list(Path('/kaggle/input').glob('**/documents.csv'))
    if not cands:
        raise FileNotFoundError('documents.csv not found — attach the competition data via the Input panel.')
    base = cands[0].parent
print('Reading from:', base)

# Load the four official competition CSVs.
documents      = pd.read_csv(base / 'documents.csv')       # knowledge base: document_id, title, text, crop
train_queries  = pd.read_csv(base / 'train_queries.csv')   # labeled queries: query_id, query
qrels          = pd.read_csv(base / 'qrels_train.csv')     # ground-truth relevance: query_id, document_id, relevance (0-3)
test_queries   = pd.read_csv(base / 'test_queries.csv')    # held-out queries to predict on

# Convenience arrays/lists built once, reused everywhere below (avoids
# repeated slow pandas .iloc / .loc lookups inside hot loops).
document_ids     = documents['document_id'].to_numpy()
documents_titles = documents['title'].fillna('').astype(str).tolist()
documents_bodies = documents['text'].fillna('').astype(str).tolist()
# crop column may be absent in some data drops -> default to empty strings so
# downstream code doesn't need to special-case a missing column.
documents_crop   = documents['crop'].fillna('').astype(str).tolist()    if 'crop'    in documents.columns else ['']*len(documents)
# The text every retriever/reranker actually "reads": title + body.
document_text    = (documents['title'].fillna('') + '. ' + documents['text'].fillna('')).astype(str)
# document_id -> row-position lookup, since IDs are not guaranteed to equal
# their row's position in the dataframe/array.
index_by_id      = {int(d): i for i, d in enumerate(document_ids)}

print('documents:', documents.shape, '| train_q:', train_queries.shape,
      '| qrels:', qrels.shape, '| test_q:', test_queries.shape)

In [ ]:
# ==========================================================================
# 2. Competition metric: nDCG@5
# ==========================================================================
# Implemented locally so we can evaluate every stage of the pipeline offline,
# without needing a leaderboard submission.

def dcg(rels, k=5):
    """Discounted Cumulative Gain over the first k relevance scores.

    rel_i is discounted by log2(rank+2) so higher-ranked hits count more.
    """
    v = np.asarray(list(rels)[:k], dtype=float)
    if len(v) == 0: return 0.0
    return float(np.sum(v / np.log2(np.arange(2, len(v) + 2))))

def evaluate_ndcg_at_5(predictions, qrels_frame=qrels):
    """predictions: dict {query_id: [doc_id, doc_id, ...]} (already ranked).

    For each query with ground-truth labels, look up the true relevance of
    the model's top-5 predicted docs, compute DCG, and normalise by the
    *ideal* DCG (i.e. what DCG would be if the top-5 truly-relevant docs
    were ranked first) to get a 0-1 nDCG@5 score. Returns the mean score
    across queries plus the full per-query score dict (handy for later
    error analysis / worst-query inspection).
    """
    # qid -> {document_id: relevance} lookup table, built once per call.
    lookup = {qid: dict(zip(g['document_id'], g['relevance']))
              for qid, g in qrels_frame.groupby('query_id')}
    scores = {}
    for qid, judged in lookup.items():
        ranked = predictions.get(qid, [])[:5]
        gains  = [judged.get(int(d), 0) for d in ranked]      # 0 if predicted doc has no/zero label
        ideal  = sorted(judged.values(), reverse=True)[:5]    # best possible top-5 ordering
        idcg   = dcg(ideal)
        scores[qid] = dcg(gains) / idcg if idcg else 0.0      # avoid divide-by-zero for queries with no relevant docs
    return float(np.mean(list(scores.values()))), scores

In [ ]:
# ==========================================================================
# 3. Shared slot extractor (crop / issue / intent / zone)
# ==========================================================================
# A small, fully hand-written information-extraction layer. It is applied
# identically to queries and to document titles, so features built from its
# output (crop match, issue match, intent match, ...) are directly
# comparable between the two. No labeled data or external gazetteer is
# needed — the vocabularies below are mined straight out of the document
# titles / crop column.

titles = documents['title'].fillna('').astype(str)

# --- Crop vocabulary -------------------------------------------------------
# Pulled from the documents' own `crop` column (excluding the generic
# "(general)" placeholder). Sorted longest-first so multi-word crop names
# (e.g. "sweet potato") are matched before shorter substrings ("potato").
CROPS = sorted({c.strip().lower() for c in documents_crop
                if c.strip() and c.strip().lower() != '(general)'}, key=len, reverse=True)

# --- Zone vocabulary --------------------------------------------------------
# Many document titles end in a parenthetical agro-ecological zone, e.g.
# "... (Northern Zone)". Extract every such trailing parenthetical.
ZONES = set()
for t in titles:
    m = re.search(r'\(([^)]+)\)\s*$', t)
    if m: ZONES.add(m.group(1).strip().lower())
ZONES = sorted(ZONES, key=len, reverse=True)

# --- Issue vocabulary --------------------------------------------------------
# Issues (pest/disease/deficiency names) are mined from document titles
# using three complementary heuristics, since there's no single title
# template that covers every case:
ISSUES = set()

# (a) "<nutrient> deficiency" patterns, e.g. "nitrogen deficiency".
for t in titles.str.lower():
    for w in re.findall(r'([a-z]+)\s+deficiency', t):
        ISSUES.add(f'{w} deficiency')

# (b) Action-verb heads followed by a noun phrase, cut off at a connective
# word. E.g. "Controlling Fall Armyworm in Maize" -> "fall armyworm".
HEADS = (r'(?:what causes|damage caused by|controlling and managing|controlling|managing|'
         r'preventing|identifying|correcting|telling|how|enhancing)')
CONN  = r'(?:\s+(?:in|on)\s+|\s+apart\b|\s+spreads\b|\s+tolerance\b)'
for t in titles:
    tl = re.sub(r'\s*\([^)]*\)\s*$', '', t).strip()   # drop trailing zone parenthetical first
    m = re.match(rf'^{HEADS}\s+(.*?){CONN}', tl, flags=re.I)
    if m:
        cand = m.group(1).strip().lower()
        if cand and 'deficiency' not in cand:          # deficiencies are already captured by rule (a)
            ISSUES.add(cand)

# (c) Two more title templates: "Adapting to X (...)" and
# "X: the risk to crops" — both name a climate/weather issue.
for t in titles.str.lower():
    m = re.match(r'adapting to (.+?)\s*\(', t)
    if m: ISSUES.add(m.group(1).strip())
    m = re.match(r'(.+?):\s*the risk to crops', t)
    if m: ISSUES.add(m.group(1).strip())

# Generic, non-specific issue words. Kept separate from "specific" issues
# because a document about "pests" in general is a much weaker match for a
# query about a *named* pest than a document naming that exact pest.
GENERIC_ISSUES = {'insect pests','pests','pest','insects','weeds','weed','disease','diseases'}
ISSUES |= GENERIC_ISSUES
ISSUES = sorted(ISSUES, key=len, reverse=True)   # longest-first, same reasoning as CROPS

# --- Intent classification ---------------------------------------------------
# Six mutually-exclusive intent categories, each with an ordered list of
# trigger phrases. Order matters: the loop below returns the *first*
# category whose trigger fires, so more specific intents should be listed
# before more generic ones that might also match part of the same text.
INTENT_ORDER = [
    ('prevention', ['prevent','avoid','protect','stop','before it hits','before it spreads','guard against']),
    ('symptom',    ['look like','looks like','what does','identify','telling','apart from look-alikes',
                    'signs of','sign of','symptom','appearance','could it be','shows spots','shows signs',
                    'how do i know','how can i tell','diagnose']),
    ('cause',      ['what causes','causes','caused by','cause of','why are','why is','why does','reason',
                    'spread','spreads','get worse','gets worse','worsen', r'make\w*\s+.*worse']),
    ('adaptation', ['cope','adapt','resilience','tolerance','build resilience','prepare for']),
    ('impact',     ['risk of','risk to','the risk','affect','impact','effect of']),
    ('treatment',  ['fertiliser','fertilizer','correct','fix','treat','control','manage','deal with',
                    'best way','remedy','recommendation','what should i do','outbreak','get rid of',
                    'how do i handle','apply','spray']),
]

def detect_intent(tl):
    """Return the first matching intent label for lowercase text `tl`, or None."""
    for label, phrases in INTENT_ORDER:
        for p in phrases:
            if '.*' in p or '\\' in p:
                if re.search(p, tl): return label   # regex-style trigger
            elif p in tl:
                return label                          # plain substring trigger
    return None

def parse_slots(text):
    """Extract {crop, issue, issue_spec, issue_gen, intent, zone} from `text`.

    Applied identically to queries and document titles so the resulting
    slot sets can be directly compared (intersected) as match features.
    """
    tl = str(text).lower()
    crop  = [c for c in CROPS  if c in tl]
    issue = [i for i in ISSUES if i in tl]
    zone  = [z for z in ZONES  if z in tl]
    # De-duplicate overlapping issue matches: drop an issue string if it is
    # a strict substring of another matched issue (keep the more specific one).
    issue = [i for i in issue if not any(i != j and i in j for j in issue)]
    spec  = [i for i in issue if i not in GENERIC_ISSUES]
    gen   = [i for i in issue if i in GENERIC_ISSUES]
    return {'crop':crop,'issue':issue,'issue_spec':spec,'issue_gen':gen,'intent':detect_intent(tl),'zone':zone}

# Pre-compute slots for every document once (reused by every query at
# feature-build and inference time — far cheaper than re-parsing titles
# per query).
doc_slots = []
for i in range(len(documents)):
    s = parse_slots(documents_titles[i])
    # Merge in the document's own structured crop column (documents have a
    # ground-truth crop tag; queries only ever have crop *inferred* from text).
    cset = set(s['crop']); cc = documents_crop[i].strip().lower()
    if cc and cc != '(general)': cset.add(cc)
    s['crop'] = list(cset); doc_slots.append(s)

# Flat lists of just the crop/issue sets per document, used for fast slot
# lookups during candidate generation (cell 6).
_doc_issue = [set(s['issue']) for s in doc_slots]
_doc_crop  = [set(s['crop'])  for s in doc_slots]

def coverage(texts, name):
    """Diagnostic: print what fraction of `texts` get a detected crop /
    issue / intent, and what fraction are "fully resolved" (have both an
    issue and an intent). Used as a data-quality check on the parser
    before relying on it for features."""
    n=len(texts); ci=ii=it=res=0
    for x in texts:
        s=parse_slots(x)
        ci+=bool(s['crop']); ii+=bool(s['issue']); it+=bool(s['intent']); res+=bool(s['issue'] and s['intent'])
    print(f'{name:6}: crop {ci/n:.0%} | issue {ii/n:.0%} | intent {it/n:.0%} | RESOLVED {res}/{n} ({res/n:.0%})')

print(f'Vocab -> crops:{len(CROPS)} issues:{len(ISSUES)} zones:{len(ZONES)}\n')
coverage(titles,'TITLES'); coverage(train_queries['query'],'TRAIN'); coverage(test_queries['query'],'TEST')

In [ ]:
# ==========================================================================
# 4. BM25 + TF-IDF
# ==========================================================================
# Two classic lexical retrieval/similarity tools:
#  - BM25 (hand-implemented): used for fast CANDIDATE GENERATION (cell 6),
#    since it can be indexed once and queried cheaply via an inverted index.
#  - TF-IDF (scikit-learn): used only as a FEATURE (cosine similarity) fed
#    into the learned rerankers, not for candidate generation itself.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

_PUNCT = re.compile(r'[^\w\s]')
def tokenize(text):
    """Lowercase, strip punctuation, whitespace-split."""
    return _PUNCT.sub(' ', str(text).lower()).split()

class BM25:
    """Minimal Okapi BM25 implementation with an inverted index.

    k1, b are the standard BM25 hyperparameters (term-frequency saturation
    and document-length normalisation strength respectively); 1.5/0.75 are
    the commonly used defaults.
    """
    def __init__(self, tokenized_docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.N = len(tokenized_docs)
        self.doc_len = np.array([len(d) for d in tokenized_docs], dtype=float)
        self.avgdl = self.doc_len.mean() if self.N else 1.0
        tfs = [Counter(d) for d in tokenized_docs]           # per-doc term frequencies
        df = Counter()
        for c in tfs:
            for w in c: df[w] += 1                            # document frequency per term
        # Standard BM25 IDF with a +0.5/+0.5 smoothing term.
        self.idf = {w: math.log(1 + (self.N - f + 0.5) / (f + 0.5)) for w, f in df.items()}
        # Inverted index: term -> list of (doc_index, term_frequency), so
        # scoring a query only touches documents that contain a query term.
        self.postings = defaultdict(list)
        for i, c in enumerate(tfs):
            for w, f in c.items(): self.postings[w].append((i, f))

    def scores(self, query_tokens):
        """Return a dense array of BM25 scores, one per document, for the
        given (already tokenized) query."""
        s = np.zeros(self.N)
        for w in set(query_tokens):
            idf = self.idf.get(w)
            if idf is None: continue                          # unseen term contributes nothing
            for i, f in self.postings[w]:
                denom = f + self.k1 * (1 - self.b + self.b * self.doc_len[i] / self.avgdl)
                s[i] += idf * f * (self.k1 + 1) / denom
        return s

# Build the BM25 index once over title+body text for every document.
bm25 = BM25([tokenize(t) for t in document_text])

# TF-IDF over the same text: unigrams+bigrams (bigrams help match short
# multi-word phrases like "fall armyworm"), English stop-words removed,
# min_df=2 to drop ultra-rare noise terms, sublinear TF scaling to dampen
# the effect of very high term counts, L2-normalised for cosine similarity.
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=2, sublinear_tf=True)
doc_tfidf = normalize(tfidf.fit_transform(document_text))
print('BM25 vocab:', len(bm25.idf), '| TF-IDF:', doc_tfidf.shape)

In [ ]:
# ==========================================================================
# 5. Rubric scorer (deterministic ceiling)
# ==========================================================================
# A fully hand-coded, deterministic relevance rule (no learned parameters).
# It encodes a plausible guess at how the competition's own relevance labels
# were likely assigned, purely from slot agreement between query and
# document. Two uses: (1) a standalone baseline ranker to sanity-check that
# learned models aren't underperforming an obvious heuristic; (2) one of the
# 16 features fed into the LightGBM reranker (cell 6).

def crop_relation(qc, dc):
    """'same' if both crop sets are empty (crop-agnostic content is
    compatible with any query) or share at least one crop; 'diff' otherwise."""
    qc, dc = set(qc), set(dc)
    if qc and dc:         return 'same' if qc & dc else 'diff'
    if not qc and not dc: return 'same'
    return 'diff'

def expected_rel(qs, ds):
    """Deterministic 0-3 relevance guess from query slots `qs` and document
    slots `ds`:
      3 -> specific issue matches, crop matches, AND intent matches
      2 -> specific issue + crop match (intent differs), OR generic issue +
           crop match with matching intent
      1 -> specific issue matches but crop differs, OR generic issue + crop
           match without matching intent
      0 -> no meaningful overlap
    """
    issue_spec = bool(set(qs['issue_spec']) & set(ds['issue_spec']))
    crop_rel   = crop_relation(qs['crop'], ds['crop'])
    intent_same = qs['intent'] is not None and qs['intent'] == ds['intent']
    if issue_spec and crop_rel == 'same' and intent_same: return 3
    if issue_spec and crop_rel == 'same':                 return 2
    if issue_spec and crop_rel == 'diff':                 return 1
    if (set(qs['issue_gen']) & set(ds['issue'])) and crop_rel == 'same':
        return 2 if intent_same else 1
    return 0

# Cache query slot-parsing results, since the same query string may be
# re-parsed many times across different stages of the pipeline.
query_slot_cache = {}
def query_slots(q):
    if q not in query_slot_cache: query_slot_cache[q] = parse_slots(q)
    return query_slot_cache[q]

def rank_by_rubric(query, k=5):
    """Rank ALL documents by (-expected_rel, -bm25_score) — rubric score is
    the primary sort key, BM25 breaks ties among equally-scored documents —
    and return the top k document IDs."""
    qs = query_slots(query)
    er = np.array([expected_rel(qs, doc_slots[i]) for i in range(len(documents))], dtype=float)
    bm = bm25.scores(tokenize(query))
    order = np.lexsort((-bm, -er))[:k]   # lexsort sorts by last key first, so -er is primary
    return document_ids[order].tolist()

# Evaluate the rubric-only ranker on the training set as a deterministic
# sanity ceiling: if learned models can't approach this, something is
# likely wrong upstream (features, training, or CV setup).
rubric_pred = {r['query_id']: rank_by_rubric(r['query']) for _, r in train_queries.iterrows()}
rubric_score, _ = evaluate_ndcg_at_5(rubric_pred)
print(f'Deterministic rubric nDCG@5 (train): {rubric_score:.4f}')

In [ ]:
# ==========================================================================
# 6. Candidate generation (BM25 top-K + slot injection) + feature builder
# ==========================================================================
RECALL_K = 150   # size of the BM25 shortlist before slot injection; tuned to balance recall vs. downstream cost

def get_candidates(query, inject=None):
    """Return (candidate_document_ids, full_bm25_score_array) for `query`.

    Candidate generation has two parts:
      1. BM25 top-RECALL_K by raw lexical score.
      2. Slot injection: any document sharing a *specific* issue or crop
         slot with the query is added even if it fell outside the BM25
         shortlist — a recall safety net for cases where wording differs
         but the underlying topic slot matches.
    `inject` optionally force-adds specific document IDs. This is ONLY used
    during TRAINING feature construction below (to guarantee every known
    positive document is represented in the training data even if BM25 and
    slot injection miss it) — it is never used at inference/test time,
    which keeps the recall check (cell 7) and CV evaluations (cells 8, 11)
    honest about real test-time recall.
    """
    bm = bm25.scores(tokenize(query))
    top = np.argsort(-bm)[:RECALL_K]
    ids = [int(document_ids[i]) for i in top]; idset = set(ids)
    qs = query_slots(query); qi, qc = set(qs['issue_spec']), set(qs['crop'])
    if qi or qc:
        for i in range(len(documents)):
            if (qi and (_doc_issue[i] & qi)) or (qc and (_doc_crop[i] & qc)):
                d = int(document_ids[i])
                if d not in idset: ids.append(d); idset.add(d)
    if inject:
        for d in inject:
            if int(d) not in idset: ids.append(int(d)); idset.add(int(d))
    return ids, bm

# The fixed feature schema used by both learned rerankers (LightGBM in
# cell 8/9, and indirectly the cross-encoder pipeline in cells 10-11 which
# consumes LightGBM's output score).
FEATURES = ['bm25','bm25_rank','tfidf_cos','expected_rel','issue_spec_match','issue_gen_match',
            'n_issue_overlap','crop_match','crop_both_general','intent_match','intent_known',
            'zone_match','title_overlap','body_overlap','q_len','title_len']

def build_rows(query, cand_ids, bm_scores, label_lookup=None):
    """Build one feature row per (query, candidate document) pair.

    label_lookup: optional {document_id: relevance} dict — if given, a
    ground-truth 'relevance' column is attached (training mode); if None,
    only features are produced (inference mode).
    """
    qs = query_slots(query); qtok = set(tokenize(query))
    # TF-IDF cosine similarity between the query and every document
    # (computed for all documents, then indexed by candidate below — the
    # TfidfVectorizer/matrix multiply is cheap enough for this).
    qv = normalize(tfidf.transform([query])); cos = (qv @ doc_tfidf.T).toarray().ravel()
    # BM25 rank (0 = highest BM25 score) among the candidate set, used as a
    # feature distinct from the raw score itself.
    order = sorted(cand_ids, key=lambda cid: -bm_scores[index_by_id[cid]])
    rank_map = {cid: r for r, cid in enumerate(order)}
    rows = []
    for cid in cand_ids:
        i = index_by_id[cid]; ds = doc_slots[i]
        ttok = set(tokenize(documents_titles[i])); btok = set(tokenize(documents_bodies[i]))
        cr = crop_relation(qs['crop'], ds['crop'])
        row = {'document_id':cid,
               'bm25':bm_scores[i],                                            # raw BM25 lexical score
               'bm25_rank':rank_map[cid],                                      # rank position by BM25 within candidates
               'tfidf_cos':cos[i],                                             # TF-IDF cosine similarity
               'expected_rel':expected_rel(qs,ds),                             # rubric-scorer feature (cell 5)
               'issue_spec_match':int(bool(set(qs['issue_spec']) & set(ds['issue_spec']))),
               'issue_gen_match':int(bool(set(qs['issue']) & set(ds['issue']))),
               'n_issue_overlap':len(set(qs['issue']) & set(ds['issue'])),
               'crop_match':int(cr=='same'),
               'crop_both_general':int((not qs['crop']) and (not ds['crop'])),
               'intent_match':int(qs['intent'] is not None and qs['intent']==ds['intent']),
               'intent_known':int(qs['intent'] is not None),
               'zone_match':int(bool(set(qs['zone']) & set(ds['zone']))),
               'title_overlap':len(qtok & ttok)/max(len(qtok),1),               # token-overlap ratio, query vs. title
               'body_overlap':len(qtok & btok)/max(len(qtok),1),                # token-overlap ratio, query vs. body
               'q_len':len(qtok),
               'title_len':len(ttok)}
        if label_lookup is not None: row['relevance'] = int(label_lookup.get(cid, 0))
        rows.append(row)
    return rows

# qid -> {document_id: relevance} lookup, used both to attach labels to
# training rows and to force-inject known positives into their candidate set.
qrels_by_q = {qid: dict(zip(g['document_id'], g['relevance'])) for qid, g in qrels.groupby('query_id')}
train_rows = []
for _, r in train_queries.iterrows():
    qid, q = r['query_id'], r['query']; labels = qrels_by_q.get(qid, {})
    # inject=labels.keys(): guarantee every labeled document for this query
    # appears in its training candidate set, so the ranker always has
    # positive examples to learn from even where BM25/slots alone would
    # have missed them.
    cand, bm = get_candidates(q, inject=list(labels.keys()))
    for row in build_rows(q, cand, bm, label_lookup=labels):
        row['query_id'] = qid; train_rows.append(row)
train_features = pd.DataFrame(train_rows)
print('train_features:', train_features.shape, '| positives:', int((train_features['relevance']>0).sum()))

In [ ]:
# ==========================================================================
# 7. Recall check (no injection = test-time realism)
# ==========================================================================
# Re-run get_candidates WITHOUT label injection — i.e. exactly as it will
# run at test time — and measure what fraction of the known-relevant
# training documents actually survive into the candidate pool. This is the
# single most important sanity check in the notebook: no reranker, however
# good, can retrieve a relevant document that never made it into the
# candidate set. A low number here would mean RECALL_K should increase or
# slot-injection logic should be broadened.
hits = tot = 0
for _, r in train_queries.iterrows():
    labels = qrels_by_q.get(r['query_id'], {}); pos = {d for d,v in labels.items() if v>0}
    if not pos: continue
    cand, _ = get_candidates(r['query']); hits += len(pos & set(cand)); tot += len(pos)
print(f'BM25 top-{RECALL_K} recall of positives: {hits}/{tot} ({hits/max(tot,1):.1%})')

In [ ]:
# ==========================================================================
# 8. LightGBM LambdaMART reranker + leak-free CV
# ==========================================================================
try:
    import lightgbm as lgb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','lightgbm','-q'], check=True)
    import lightgbm as lgb
from sklearn.model_selection import GroupKFold

def make_ranker():
    """LightGBM configured for LEARNING TO RANK (LambdaMART), not plain
    regression/classification. label_gain=[0,1,2,3] matches the 0-3
    relevance scale used by qrels. Hyperparameters are deliberately
    conservative (shallow trees via num_leaves=15, min_child_samples=10,
    heavy row/column subsampling, reg_lambda=0.5) since the number of
    labeled queries is small and this reduces overfitting risk."""
    return lgb.LGBMRanker(objective='lambdarank', n_estimators=250, learning_rate=0.05,
                          num_leaves=15, min_child_samples=10, subsample=0.9, subsample_freq=1,
                          colsample_bytree=0.8, reg_lambda=0.5, label_gain=[0,1,2,3], verbosity=-1)

def cv_lgb(features_df, n_splits=5):
    """Leak-free cross-validation of the LightGBM reranker alone.

    GroupKFold is grouped by query_id, so every row belonging to one query
    stays in the same fold — a query's candidates are never split across
    train/validation. For each fold: train on the other folds' rows, then
    generate validation predictions by re-running get_candidates/build_rows
    WITHOUT label injection (i.e. test-time realism), so the evaluation
    reflects genuine candidate generation rather than the label-injected
    training candidate set.
    """
    gkf = GroupKFold(n_splits=n_splits); preds = {}
    qids = features_df['query_id'].values
    for tr_idx, va_idx in gkf.split(features_df, groups=qids):
        tr = features_df.iloc[tr_idx].sort_values('query_id')
        groups = tr.groupby('query_id', sort=False).size().tolist()   # LightGBM needs group sizes, not IDs, for ranking loss
        model = make_ranker(); model.fit(tr[FEATURES], tr['relevance'], group=groups)
        for qid in features_df.iloc[va_idx]['query_id'].unique():
            q = train_queries.loc[train_queries['query_id']==qid,'query'].values[0]
            cand, bm = get_candidates(q); vf = pd.DataFrame(build_rows(q, cand, bm))
            vf['score'] = model.predict(vf[FEATURES])
            preds[qid] = vf.sort_values('score', ascending=False)['document_id'].head(5).tolist()
    return evaluate_ndcg_at_5(preds)

# This is the first HONEST (non-ceiling, non-optimistic) estimate of how
# well the learned reranker generalises to unseen queries.
lgb_cv, _ = cv_lgb(train_features)
print(f'LightGBM leak-free CV nDCG@5: {lgb_cv:.4f}')

In [ ]:
# ==========================================================================
# 9. Train final LightGBM on ALL training queries
# ==========================================================================
# Cross-validation above has already given an honest performance estimate,
# so now train one production model on the full training set (more data
# generally helps) for use in the submission pipeline (cells 10-12).
tr = train_features.sort_values('query_id')
groups = tr.groupby('query_id', sort=False).size().tolist()
final_model = make_ranker()
final_model.fit(tr[FEATURES], tr['relevance'], group=groups)
print('final_model trained.')

In [ ]:
# ==========================================================================
# 10. Cross-encoder (bge-reranker-base) fine-tune + retrieve_ce
# ==========================================================================
# Second, more powerful reranking stage: a pretrained cross-encoder that
# scores a (query, document) pair JOINTLY (unlike BM25/TF-IDF which score
# them independently), fine-tuned directly on this task's qrels.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # single GPU -> avoids DataParallel bug
try:
    from sentence_transformers.cross_encoder import CrossEncoder
    from sentence_transformers import InputExample
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','sentence-transformers','-q'], check=True)
    from sentence_transformers.cross_encoder import CrossEncoder
    from sentence_transformers import InputExample
from torch.utils.data import DataLoader

CE_BASE       = 'BAAI/bge-reranker-base'   # pretrained cross-encoder checkpoint (downloaded from Hugging Face)
CE_EPOCHS     = 3
CE_BATCH      = 16
FIRST_STAGE_N = 50   # only the LightGBM top-50 candidates get scored by the (expensive) cross-encoder

# O(1) lookup of a document's full text (title+body) by ID — needed
# repeatedly when building (query, document) pairs for the cross-encoder.
doc_text_by_id = {int(d): document_text.iloc[index_by_id[int(d)]] for d in document_ids}

def train_ce(train_qids=None):
    """Fine-tune a fresh CrossEncoder on qrels.

    train_qids: optional subset of query_ids to restrict training to (used
    for fold-specific fine-tuning in cell 11's leak-free CV, so a fold's
    held-out queries are never seen during that fold's fine-tuning).
    Relevance labels (0-3) are rescaled to 0-1 since the model is a
    regression head with num_labels=1.
    """
    ex = []
    for _, r in qrels.iterrows():
        if train_qids is not None and r['query_id'] not in train_qids: continue
        q = train_queries.loc[train_queries['query_id']==r['query_id'],'query']
        if q.empty: continue
        ex.append(InputExample(texts=[q.values[0], doc_text_by_id[int(r['document_id'])]],
                               label=float(r['relevance'])/3.0))
    m = CrossEncoder(CE_BASE, num_labels=1, max_length=256)
    if hasattr(m.model,'module'): m.model = m.model.module   # unwrap DataParallel if it got auto-wrapped
    m.fit(train_dataloader=DataLoader(ex, shuffle=True, batch_size=CE_BATCH),
          epochs=CE_EPOCHS, warmup_steps=100, show_progress_bar=False)
    return m

# Fine-tune on the FULL training set — this `ce` model (together with
# `final_model` from cell 9) is what actually generates the submission.
ce = train_ce()
print('Cross-encoder fine-tuned on all qrels.')

def retrieve_ce(query, k=5, model=None, first_stage_n=FIRST_STAGE_N):
    """Full two-stage inference: BM25/slot candidates -> LightGBM rerank ->
    keep top `first_stage_n` -> cross-encoder rerank -> return top `k`.
    This exact function is used both for the leak-free CV gate (cell 11,
    with fold-specific models passed in) and for the final submission
    (cell 12, with the full-data final_model/ce).
    """
    model = model or ce
    cand, bm = get_candidates(query)
    vf = pd.DataFrame(build_rows(query, cand, bm))
    vf['lgb'] = final_model.predict(vf[FEATURES])
    sl = vf.sort_values('lgb', ascending=False).head(first_stage_n).copy()   # narrow to top-50 before the costly CE pass
    pairs = [[query, doc_text_by_id[int(d)]] for d in sl['document_id']]
    sl['ce'] = model.predict(pairs, show_progress_bar=False)
    return sl.sort_values('ce', ascending=False)['document_id'].head(k).tolist()

# NOTE: this in-sample score is OPTIMISTIC — the cross-encoder has already
# seen these exact training queries/labels during fine-tuning above, so
# this is a smoke test that the wiring works, NOT a valid generalisation
# estimate. The honest estimate comes from cell 11.
ce_pred = {r['query_id']: retrieve_ce(r['query']) for _, r in train_queries.iterrows()}
ce_insample, _ = evaluate_ndcg_at_5(ce_pred)
print(f'In-sample nDCG@5 (optimistic): {ce_insample:.4f}')

In [ ]:
# ==========================================================================
# 11. LEAK-FREE CV of full LightGBM -> Cross-encoder  (THE GATE)
#     Uses the SAME epochs/shortlist as the shipped model.
# ==========================================================================
# The most important cell for deciding whether to submit. Unlike cell 10's
# in-sample check, this fine-tunes a FRESH cross-encoder per fold using
# ONLY that fold's training queries — essential to avoid leakage, since a
# cross-encoder trained on the held-out queries' own labels would otherwise
# "already know the answer" when evaluated on them.

def cv_full_pipeline(n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    tq = train_queries.reset_index(drop=True); preds = {}
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(tq, groups=tq['query_id'])):
        tr_qids = set(tq.iloc[tr_idx]['query_id']); va_qids = set(tq.iloc[va_idx]['query_id'])
        print(f'--- Fold {fold+1}/{n_splits}: {len(tr_qids)} train, {len(va_qids)} val ---')
        # Fold-specific LightGBM: trained only on this fold's training rows.
        tf = train_features[train_features['query_id'].isin(tr_qids)].sort_values('query_id')
        g  = tf.groupby('query_id', sort=False).size().tolist()
        lgb_fold = make_ranker(); lgb_fold.fit(tf[FEATURES], tf['relevance'], group=g)
        # Fold-specific cross-encoder: fine-tuned only on this fold's
        # training query_ids, so it has never seen this fold's validation
        # queries during fine-tuning.
        ce_fold = train_ce(train_qids=tr_qids)
        for qid in va_qids:
            q = train_queries.loc[train_queries['query_id']==qid,'query'].values[0]
            cand, bm = get_candidates(q); vf = pd.DataFrame(build_rows(q, cand, bm))
            vf['lgb'] = lgb_fold.predict(vf[FEATURES])
            sl = vf.sort_values('lgb', ascending=False).head(FIRST_STAGE_N).copy()
            pairs = [[q, doc_text_by_id[int(d)]] for d in sl['document_id']]
            sl['ce'] = ce_fold.predict(pairs, show_progress_bar=False)
            preds[qid] = sl.sort_values('ce', ascending=False)['document_id'].head(5).tolist()
    return evaluate_ndcg_at_5(preds)

ce_cv_score, _ = cv_full_pipeline()
print(f'\n=== LEAK-FREE CV (LightGBM -> Cross-encoder): {ce_cv_score:.4f} ===')
# Decision rule: only ship the (much more expensive) cross-encoder stage if
# its honest CV score beats both the LightGBM-only CV score AND the
# current leaderboard score — otherwise the added complexity/training time
# isn't paying for itself and the simpler LightGBM-only submission should
# be preferred.
print(f'LightGBM alone: {lgb_cv:.4f}. Submit only if this beats your current LB (0.92).')

In [ ]:
# ==========================================================================
# 12. Build submission (LightGBM -> Cross-encoder)
# ==========================================================================
# Run the full two-stage pipeline (full-data final_model + ce, NOT
# fold-specific models) over every test query, collecting the top-5
# predicted document IDs per query in the exact format the competition
# expects: one row per (query, predicted document), 5 rows per query,
# ranked best-first.
rows = []
for qid in test_queries['query_id']:
    q = test_queries.loc[test_queries['query_id']==qid,'query'].values[0]
    for d in retrieve_ce(q, k=5):
        rows.append({'QueryId': qid, 'DocumentId': int(d)})
submission = pd.DataFrame(rows, columns=['QueryId','DocumentId'])

# --- Structural validation before writing the file -------------------------
expected_order = test_queries['query_id'].repeat(5).tolist()
assert list(submission.columns) == ['QueryId','DocumentId']                         # exact column names/order
assert len(submission) == len(test_queries) * 5, len(submission)                    # 5 rows per test query
assert submission['QueryId'].tolist() == expected_order, 'query/rank order changed'  # query order preserved
assert submission.groupby('QueryId', sort=False).size().eq(5).all()                 # exactly 5 predictions per query
assert not submission.duplicated(['QueryId','DocumentId']).any(), 'duplicate doc in a query'  # no duplicate docs
assert set(submission['DocumentId']).issubset(set(int(d) for d in document_ids))    # every doc id is real

submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Saved /kaggle/working/submission.csv', submission.shape)
submission.head(10)

---

## Acknowledgements

This notebook was built as part of the **AI Saturdays Lagos** cohort
program for the *Agricultural Extension RAG: Smart Retrieval for Farmers*
challenge.

**Thanks to:**

- **Our mentors**, for guidance on the retrieve-then-rerank architecture,
  leak-free evaluation practices, and general feedback throughout the
  cohort.
- **AI Saturdays Lagos** and the challenge organizers, for designing the
  dataset, the nDCG@5 evaluation setup, and the cohort structure that
  made this project possible.
- **The open-source ecosystem** this pipeline stands on: `scikit-learn`
  (TF-IDF), `LightGBM` (LambdaMART ranking), `sentence-transformers` and
  the `BAAI/bge-reranker-base` model (cross-encoder reranking), and the
  broader `pandas`/`numpy` stack.
- **Our teammates**, for contributions across data exploration, feature
  design (the slot extractor and rubric scorer), model training, and
  evaluation.

_Add specific names below for the final submission:_

- **Team members:** _Hamna Kaleem, Kamaya Ndigwa Espérance Martine_
- **Mentor(s):** _<mentor name(s)>_
